# Anthony: first GD / SGD baseline

**Goal:** use the same prepared data to compare our two working algorithms.

Start with **Restart Kernel and Run All**. This loads training data only; it does not use the competition test set or make a submission.

This adapts Shu's smaller baseline: 5 survey-presence fields + 111 ordered/numeric fields, then an intercept. It excludes his categorical encoding and is not a reproduction of his best result. Five missing-value codes have been corrected. Other inherited feature choices remain candidates for later review.


In [1]:
from pathlib import Path
import sys
import time
import numpy as np

# Works when launched from the project root or experiments/.
ROOT = Path.cwd()
if not (ROOT / "implementations.py").exists():
    ROOT = ROOT.parent
assert (ROOT / "implementations.py").exists(), "Open this notebook inside the project."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from implementations import mean_squared_error_gd, mean_squared_error_sgd
from preprocessing.preprocess import train_val_split
from preprocessing.anthony_baseline import (
    FEATURE_NAMES, load_training_data, fit_preprocessor,
    transform_features, classification_metrics,
)

print("Python:", sys.executable)
print("NumPy:", np.__version__)


Python: /opt/homebrew/anaconda3/envs/project1-work/bin/python
NumPy: 1.23.1


## 1. Load and split

Keep 25% aside for validation, using seed 42 to match Shu's split. Each row is a person; each column is a selected feature. Labels stay **-1 / +1** for our MSE methods.

The loader checks that the feature rows and label rows have matching IDs. It only loads the 116 selected columns, in small chunks to limit memory use.


In [2]:
started = time.perf_counter()
x, y = load_training_data(ROOT / "data")
x_train, x_val, y_train, y_val = train_val_split(x, y, val_ratio=0.25, seed=42)
del x, y
print("Training:", x_train.shape, "Validation:", x_val.shape)
print(f"Positive labels: train {np.mean(y_train == 1):.2%}, validation {np.mean(y_val == 1):.2%}")
print(f"Loaded and split in {time.perf_counter() - started:.1f}s")


Training: (246102, 116) Validation: (82033, 116)
Positive labels: train 8.83%, validation 8.82%
Loaded and split in 11.2s


## 2. Prepare using training statistics only


1. Interpret fixed survey codes: for example, some codes mean 'unknown', not a measurement.
2. Fill missing ordered values with their **training-column mean**.
3. Scale ordered columns using their **training mean and standard deviation**.
4. Add a column of ones so the model can learn an intercept.

Validation reuses those exact training statistics. Constant columns cannot cause division by zero; entirely missing training columns contribute zero. The first five fields retain Shu's original observed-code / missing (-1) encoding.

The five corrected missing codes are `MAXVO2_`/`FC60_`: 999, and `PAFREQ1_`/`PAFREQ2_`/`STRFREQ_`: 99. These corrections do not establish that validation F1 will improve; that requires a matched experiment.


In [3]:
tx_train, preprocessing_state = fit_preprocessor(x_train)
tx_val = transform_features(x_val, preprocessing_state)
del x_train, x_val
print("Prepared:", tx_train.shape, tx_val.shape)
print("All finite:", np.isfinite(tx_train).all() and np.isfinite(tx_val).all())
print("Entirely missing training columns:", np.sum(preprocessing_state["all_missing"]))


Prepared: (246102, 117) (82033, 117)
All finite: True
Entirely missing training columns: 0


## 3. Train GD

Use 300 updates for this first run. The step size below is computed from the training matrix to keep the quadratic-loss updates stable. It is a starting setting, not an optimized choice.

Each GD update reads every training row.


In [4]:
initial_w = np.zeros(tx_train.shape[1])
# Largest curvature of half-MSE; 1/L is a conservative full-batch step.
gram = tx_train.T @ tx_train / len(y_train)
L = np.linalg.eigvalsh(gram)[-1]
gd_gamma = 1.0 / L
gd_iters = 300
started = time.perf_counter()
w_gd, loss_gd = mean_squared_error_gd(y_train, tx_train, initial_w, gd_iters, gd_gamma)
gd_seconds = time.perf_counter() - started
assert np.isfinite(w_gd).all() and np.isfinite(loss_gd)
print(f"GD: steps={gd_iters}, gamma={gd_gamma:.6g}, training half-MSE={loss_gd:.6f}, time={gd_seconds:.2f}s")


GD: steps=300, gamma=0.12577, training half-MSE=0.142465, time=3.38s


## 4. Train SGD

Use three times the number of training rows as the update count. Sampling is with replacement: this is three dataset-sized amounts of work, **not** three guaranteed passes through every row.

For this first run, use a conservative step based on the largest training-row squared length. This avoids a large row causing an unstable update, but may learn slowly. Seed 42 makes the sampled sequence repeatable. SGD and GD have different budgets here; this is not a matched-compute speed benchmark.


In [5]:
max_row_squared_length = np.max(np.einsum("ij,ij->i", tx_train, tx_train))
sgd_gamma = 1.0 / max_row_squared_length
sgd_iters = 3 * len(y_train)
np.random.seed(42)
started = time.perf_counter()
w_sgd, loss_sgd = mean_squared_error_sgd(y_train, tx_train, initial_w, sgd_iters, sgd_gamma)
sgd_seconds = time.perf_counter() - started
assert np.isfinite(w_sgd).all() and np.isfinite(loss_sgd)
print(f"SGD: steps={sgd_iters}, gamma={sgd_gamma:.6g}, training half-MSE={loss_sgd:.6f}, time={sgd_seconds:.2f}s")


SGD: steps=738306, gamma=1.96501e-06, training half-MSE=0.143493, time=2.32s


## 5. Compare on validation

Our linear models output scores, not probabilities. Convert scores to +1 when they are at least a chosen threshold, otherwise -1.

Report two thresholds: **0** (the natural midpoint of -1/+1) and **-0.55** (borrowed from Shu's smaller baseline). We are not claiming either is optimal. Accuracy can look high simply by predicting the majority class; F1 measures positive-class performance.

These are holdout results, not AIcrowd scores. Later tuning on this same validation set makes it less independent, so record how settings were chosen.


In [6]:
results = []
negative = classification_metrics(y_val, -np.ones_like(y_val))
results.append({"method": "Always -1", "threshold": None, "half_mse": None, **negative})
for name, w in [("GD", w_gd), ("SGD", w_sgd)]:
    scores = tx_val @ w
    half_mse = 0.5 * np.mean((scores - y_val) ** 2)
    for threshold in (0.0, -0.55):
        predictions = np.where(scores >= threshold, 1, -1)
        results.append({"method": name, "threshold": threshold, "half_mse": float(half_mse),
                        **classification_metrics(y_val, predictions)})

print(f"{'Method':<12} {'Threshold':>10} {'Val half-MSE':>14} {'Val F1':>10} {'Val accuracy':>14}")
for row in results:
    threshold = '-' if row['threshold'] is None else f"{row['threshold']:.2f}"
    mse = '-' if row['half_mse'] is None else f"{row['half_mse']:.6f}"
    print(f"{row['method']:<12} {threshold:>10} {mse:>14} {row['f1']:>10.4f} {row['accuracy']:>14.4f}")


Method        Threshold   Val half-MSE     Val F1   Val accuracy
Always -1             -              -     0.0000         0.9118
GD                 0.00       0.142665     0.0017         0.9118
GD                -0.55       0.142665     0.3433         0.8770
SGD                0.00       0.143658     0.0017         0.9118
SGD               -0.55       0.143658     0.3277         0.8788


## 6. Save the handoff results

Record the settings and validation metrics from this run for the report. The JSON contains aggregate results only, not raw data or predictions. Timings vary with the machine and are not a matched-compute benchmark.


In [7]:
import json
import platform

record = {
    "experiment": "Anthony GD/SGD smaller-feature baseline",
    "python": platform.python_version(),
    "numpy": np.__version__,
    "split": {"seed": 42, "validation_fraction": 0.25,
              "training_rows": len(y_train), "validation_rows": len(y_val),
              "stratified": False},
    "features": {"source_columns": len(FEATURE_NAMES),
                 "model_columns_including_intercept": tx_train.shape[1],
                 "one_hot": False,
                 "missing_code_corrections": {
                     "MAXVO2_": 999, "FC60_": 999,
                     "PAFREQ1_": 99, "PAFREQ2_": 99, "STRFREQ_": 99}},
    "models": {
        "GD": {"updates": gd_iters, "gamma": float(gd_gamma),
               "gamma_rule": "1 / largest eigenvalue of X.T @ X / n",
               "training_half_mse": float(loss_gd), "seconds": gd_seconds},
        "SGD": {"updates": sgd_iters, "gamma": float(sgd_gamma), "seed": 42,
                "sampling": "one row per update, uniform with replacement",
                "gamma_rule": "1 / maximum squared training-row length",
                "training_half_mse": float(loss_sgd), "seconds": sgd_seconds}},
    "validation": results,
    "notes": ["Initial weights are all zero; labels are -1/+1.",
              "Training-fitted filling and scaling are reused on validation.",
              "Threshold -0.55 is inherited from Shu, not tuned here.",
              "These are local holdout results, not AIcrowd scores."]
}
result_path = ROOT / "experiments" / "anthony_baseline_results.json"
result_path.write_text(json.dumps(record, indent=2, allow_nan=False) + "\n")
print("Saved", result_path.name)


Saved anthony_baseline_results.json


## 7. Follow-up: adding categorical features to GD

The standalone comparison in [gd_onehot_comparison/](gd_onehot_comparison/README.md) keeps this split and preprocessing, then adds Shu's categorical fields. It also checks longer training and threshold selection separately.

At the same fixed threshold **-0.55**, longer training gave F1 **0.3442** for this smaller GD baseline and **0.4300** after adding one-hot features. Both reached the chosen gradient tolerance. These results come from that separate script; this notebook retains the original GD/SGD experiment.

See [the experiment guide](anthony_README.md) and [recorded comparison metrics](gd_onehot_comparison/recorded_results.json) for reproducible settings and results. All results are local validation measurements, not AIcrowd scores.
